# Sparkle V2 — Fase 0 + 1: Setup del entorno y obtención del dataset DAiSEE

Este notebook arranca desde cero. Al terminar vas a tener:
1. El entorno de Colab configurado (GPU + librerías).
2. Google Drive montado para persistir resultados (los features, no los videos crudos).
3. El dataset **DAiSEE** descargado (vía mirror de Kaggle) y su estructura explorada.

**Antes de correr nada:** andá a `Entorno de ejecución > Cambiar tipo de entorno de ejecución` y elegí **GPU (T4)**.

## 1. Verificar GPU

In [1]:
!nvidia-smi

Tue Jul  7 15:16:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Si esto tira error, es porque no activaste el runtime con GPU. Volvé a `Entorno de ejecución > Cambiar tipo de entorno de ejecución` y elegí GPU antes de seguir.

## 2. Instalar dependencias

Colab ya trae `tensorflow`, `numpy`, `pandas`, `opencv-python` preinstalados en la mayoría de los runtimes. Instalamos lo que falta: `mediapipe` y `kaggle`.

In [2]:
!pip install -q mediapipe kaggle

import mediapipe as mp
import cv2
import pandas as pd
import numpy as np
import tensorflow as tf

print('MediaPipe:', mp.__version__)
print('OpenCV:', cv2.__version__)
print('TensorFlow:', tf.__version__)
print('GPU disponible para TF:', tf.config.list_physical_devices('GPU'))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 122.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 13.9 MB/s eta 0:00:00
MediaPipe: 0.10.35
OpenCV: 4.13.0
TensorFlow: 2.20.0
GPU disponible para TF: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 3. Montar Google Drive

Vamos a usar Drive solo para guardar cosas **livianas**: los landmarks extraídos (números, no imágenes), los CSVs de labels, y checkpoints del modelo. Los frames de video crudos se procesan en el disco efímero de Colab (`/content/`) y se descartan, así no gastamos cuota de Drive ni mezclamos con la política de privacidad que definiste en la tesis (procesamiento local, sin persistir video).

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/SparkleV2'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/features', exist_ok=True)   # landmarks extraídos (liviano)
os.makedirs(f'{PROJECT_DIR}/labels', exist_ok=True)      # CSVs de labels de DAiSEE
os.makedirs(f'{PROJECT_DIR}/checkpoints', exist_ok=True) # modelos entrenados
print('Carpetas creadas en:', PROJECT_DIR)

Mounted at /content/drive
Carpetas creadas en: /content/drive/MyDrive/SparkleV2


## 4. Configurar acceso a Kaggle

El dataset oficial de DAiSEE se pide por formulario (people.iith.ac.in/vineethnb/resources/daisee) y puede tardar días en aprobarse. Mientras tanto, arrancamos con el **mirror en Kaggle** (`olgaparfenova/daisee`), que tiene el mismo contenido.

**Pasos para conseguir tu `kaggle.json`** (una sola vez):
1. Andá a https://www.kaggle.com y creá una cuenta gratis si no tenés.
2. Entrá a tu perfil → `Settings` → sección `API` → botón **`Create New Token`**.
3. Se descarga un archivo `kaggle.json` a tu compu.
4. Ejecutá la celda de abajo: te va a pedir que subas ese archivo.

In [4]:
from google.colab import files
import os

print('Subí tu archivo kaggle.json:')
uploaded = files.upload()

os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/kaggle.json
!chmod 600 /root/.kaggle/kaggle.json
print('Listo, credenciales configuradas.')

Subí tu archivo kaggle.json:


Saving kaggle.json to kaggle.json
Listo, credenciales configuradas.


## 5. Descargar el dataset DAiSEE

Ojo: el dataset completo (videos) pesa varios GB. Lo bajamos al disco efímero de Colab (`/content/daisee_raw`), no a Drive, porque no lo necesitamos persistido — solo necesitamos los *features* que extraigamos de él (Fase 3).

In [5]:
os.makedirs('/content/daisee_raw', exist_ok=True)
!kaggle datasets download -d olgaparfenova/daisee -p /content/daisee_raw --unzip

Dataset URL: https://www.kaggle.com/datasets/olgaparfenova/daisee
License(s): unknown
100% 14.3G/14.3G [02:57<00:00, 86.2MB/s]



Si esta celda falla con un error de autenticación (401/403), lo más probable es que el `kaggle.json` no se haya subido bien — volvé a la celda 4.

Si falla porque el dataset no existe con ese identificador, buscamos alternativas: correlo y pegame el error exacto que tira.

## 6. Explorar la estructura del dataset

DAiSEE típicamente viene organizado así:
```
DataSet/
  Train/  Validation/  Test/   (carpetas por participante, con clips .avi/.mp4)
Labels/
  TrainLabels.csv  ValidationLabels.csv  TestLabels.csv
```
Vamos a listar lo que efectivamente bajó, porque el mirror de Kaggle a veces reorganiza las carpetas.

In [6]:
for root, dirs, filenames in os.walk('/content/daisee_raw'):
    level = root.replace('/content/daisee_raw', '').count(os.sep)
    if level > 2:
        continue
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    if level == 2:
        sub_indent = '  ' * (level + 1)
        for f in filenames[:5]:
            print(f'{sub_indent}{f}')
        if len(filenames) > 5:
            print(f'{sub_indent}... ({len(filenames)} archivos en total)')

daisee_raw/
  DAiSEE/
    DataSet/
      Train.txt
      Validation.txt
      Test.txt
    Labels/
      AllLabels.csv
      TestLabels.csv
      ValidationLabels.csv
      TrainLabels.csv
    GenderClips/
      Males
      Females


In [7]:
# Buscar los CSVs de labels donde sea que hayan quedado
import glob
csv_files = glob.glob('/content/daisee_raw/**/*.csv', recursive=True)
print('CSVs encontrados:')
for c in csv_files:
    print(' -', c)

CSVs encontrados:
 - /content/daisee_raw/DAiSEE/Labels/AllLabels.csv
 - /content/daisee_raw/DAiSEE/Labels/TestLabels.csv
 - /content/daisee_raw/DAiSEE/Labels/ValidationLabels.csv
 - /content/daisee_raw/DAiSEE/Labels/TrainLabels.csv


In [8]:
# Inspeccionar el primer CSV de labels que aparezca
if csv_files:
    df_sample = pd.read_csv(csv_files[0])
    print('Archivo:', csv_files[0])
    print('Shape:', df_sample.shape)
    display(df_sample.head())
    print('\nColumnas:', list(df_sample.columns))
else:
    print('No se encontraron CSVs todavía — revisemos la estructura de carpetas manualmente.')

Archivo: /content/daisee_raw/DAiSEE/Labels/AllLabels.csv
Shape: (8925, 5)


,ClipID,Boredom,Engagement,Confusion,Frustration
0,1100011002.avi,0,2,0,0
1,1100011003.avi,0,2,0,0
2,1100011004.avi,0,3,0,0
3,1100011005.avi,0,3,0,0
4,1100011006.avi,0,3,0,0



Columnas: ['ClipID', 'Boredom', 'Engagement', 'Confusion', 'Frustration ']


In [9]:
# Buscar los archivos de video
video_files = glob.glob('/content/daisee_raw/**/*.avi', recursive=True) + \
              glob.glob('/content/daisee_raw/**/*.mp4', recursive=True)
print(f'Videos encontrados: {len(video_files)}')
if video_files:
    print('Ejemplo:', video_files[0])
    cap = cv2.VideoCapture(video_files[0])
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    print(f'FPS: {fps}, Frames: {frame_count}, Resolución: {width}x{height}')
    cap.release()

Videos encontrados: 9068
Ejemplo: /content/daisee_raw/DAiSEE/DataSet/Validation/410027/4100272051/4100272051.avi
FPS: 30.0, Frames: 300, Resolución: 640x480


## 7. Copiar los CSVs de labels a Drive (persistencia liviana)

In [10]:
import shutil
for c in csv_files:
    shutil.copy(c, f'{PROJECT_DIR}/labels/')
print('Labels copiados a Drive:', os.listdir(f'{PROJECT_DIR}/labels'))

Labels copiados a Drive: ['AllLabels.csv', 'TestLabels.csv', 'ValidationLabels.csv', 'TrainLabels.csv']


## Checkpoint de la Fase 0+1

Si llegaste hasta acá con las celdas corriendo sin error, ya tenés:
- ✅ Entorno con GPU y librerías listas.
- ✅ Drive montado con estructura de carpetas del proyecto.
- ✅ Dataset DAiSEE descargado y explorado.
- ✅ Labels persistidos en Drive.

**Pegame acá el output de las celdas 6, y 7ª (el head del CSV y la cantidad de videos encontrados)** para confirmar que la estructura coincide con lo esperado antes de armar la Fase 2 (extracción de frames + MediaPipe), porque el mirror de Kaggle a veces cambia nombres de columnas o la jerarquía de carpetas respecto del dataset oficial.